[<img src="../quantumsymmetry_logo.png" alt="QuantumSymmetry" width="450"/>](https://github.com/dariopicozzi/quantumsymmetry)

> **Note:** if you are running this notebook on Google Colab, the next cell will install `quantumsymmetry` and its dependencies:

In [1]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip -q install quantumsymmetry

# Periodic systems and crystals

Up to here every example has been a single molecule sitting in vacuum, with a finite point group of geometrical symmetries. **Crystalline materials** are infinite periodic arrangements of atoms, and their symmetry group — the **space group** — combines point-group operations with discrete lattice translations.

`quantumsymmetry` extends the symmetry-adapted encoding to periodic systems via the `PeriodicEncoding` class. The method is described in the article:

> *Picozzi, D. (manuscript). Periodic symmetry-adapted encoding: qubit reduction in crystalline electronic structure.*

Conceptually, three new ingredients appear compared to the molecular case:

1. **Supercells and $k$-point folding.** A periodic Hartree-Fock calculation in a Gaussian-orbital basis is most naturally expressed in a Bloch basis at multiple $k$-points; folding this $N_k$-point calculation into a $\Gamma$-point supercell Hamiltonian gives an object that the SAE machinery can chew on.
2. **Active bands rather than active atomic orbitals.** The active space is selected from a window of bands around the Fermi level, exactly the periodic analogue of choosing frozen-core/active/virtual orbitals around the molecular HOMO-LUMO gap.
3. **Crystal translations as Boolean symmetries.** In a supercell that contains $N_k$ primitive cells, certain half-lattice translations map the supercell to itself modulo a supercell vector. These are exact $\mathbb{Z}_2$ symmetries of the folded Hamiltonian, and each one removes one extra qubit on top of what spin parity and point-group symmetries would give.

We illustrate all of this on a concrete example: caesium chloride (CsCl) in its B2 ($Pm\overline{3}m$) crystal structure, which reaches the full theoretical reduction of eight Boolean generators on a small active space.

1. [CsCl in a (2,2,2) supercell](#CsCl)
2. [The generators that get detected](#generators)
3. [The reduced Hamiltonian and its spectrum](#hamiltonian)


<a name="CsCl"></a>
## CsCl in a (2,2,2) supercell

CsCl is a textbook ionic crystal in the B2 structure: a simple cubic lattice with Cs at the origin and Cl at the body-centred position $(a/2, a/2, a/2)$, where $a = 4.123$ Å is the experimental lattice constant. The space group is $Pm\overline{3}m$, which combines lattice translations with the full cubic point group.

We build a $(2,2,2)$ supercell, which contains $2^3 = 8$ primitive cells. In the figure below, the black cube is the supercell itself, the thin grey lines outline its 8 internal primitive cells, and one of them is highlighted in translucent red. Cs atoms (purple) sit at the cube corners and Cl atoms (green) at the body centres; the Cs–Cl nearest-neighbour pairs are drawn as thin sticks so the lattice network is visible.

<p align="center" style="background-color:white; padding:8px;"><img src="figures_periodic/supercell_cscl_primitive_cells.png" alt="CsCl supercell with 8 primitive cells" width="380"/></p>

Inside this supercell we select a **CAS(6,7)** active space: 6 electrons in 7 active spatial orbitals, taken as the three HOMOs (MOs 62–64, a $\Gamma_4^-$ triplet), the LUMO (MO 65, $\Gamma_1^+$), and the next virtual triplet (MOs 66–68, $X_1^+$). That gives 14 active spin-orbitals, so the Jordan-Wigner baseline is 14 qubits.

Translating every atom by the half-lattice vector $\mathbf{T} = \mathbf{a}_0/2$ permutes the atoms onto each other modulo a supercell lattice vector, so $\mathbf{T}$ is an exact symmetry of the periodic Hamiltonian. The arrow below marks one such half-translation along the $\mathbf{a}_0$ direction; the two analogous translations along $\mathbf{a}_1$ and $\mathbf{a}_2$ are also symmetries of this cubic supercell:

<p align="center" style="background-color:white; padding:8px;"><img src="figures_periodic/supercell_cscl_half_translation.png" alt="half-lattice translation along a_0/2" width="380"/></p>

Each half-translation generates a $\mathbb{Z}_2$ operator that removes one qubit from the encoding, in the same way that point-group symmetries do for molecules. We pass the active space to `PeriodicEncoding` as a list of 1-based MO indices:

In [2]:
import io, contextlib
import numpy as np
from quantumsymmetry import PeriodicEncoding

a_lat = 4.123  # angstrom, experimental CsCl lattice constant
# Simple cubic primitive lattice vectors
a = np.diag([a_lat, a_lat, a_lat])
atom = [
    ['Cs', (0.0,      0.0,      0.0     )],
    ['Cl', (a_lat/2., a_lat/2., a_lat/2.)],
]

# CAS(6,7): MOs 62-68 (1-based) — HOMO triplet, LUMO, next virtual triplet
active_mos = list(range(62, 69))

# Silence the PySCF/build progress lines so the cell output stays compact
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    encoding = PeriodicEncoding(
        atom = atom, a = a, basis = 'gth-szv-molopt-sr', pseudo = 'gth-pade',
        kpts = (2, 2, 2),
        active_mos = active_mos,
        name = 'CsCl-222-CAS67',
    )

print(f'Spin-orbitals       : {encoding.nspinorbital}')
print(f'Hartree-Fock (Nα,Nβ): ({encoding.nelectron_up}, {encoding.nelectron_down})')
print(f'Number of generators: {len(encoding.symmetry_generators)}')
print(f'Qubits after reduction: '
      f'{encoding.nspinorbital} -> '
      f'{encoding.nspinorbital - len(encoding.target_qubits)}')

Spin-orbitals       : 14
Hartree-Fock (Nα,Nβ): (3, 3)
Number of generators: 8
Qubits after reduction: 14 -> 6


The build takes about ten seconds — most of it is the periodic Hartree-Fock at 8 $k$-points and the automatic detection of space-group $\mathbb{Z}_2$ generators. The output should report that the 14 active spin-orbitals collapse to **6 qubits**.

<a name="generators"></a>
## The generators that get detected

`PeriodicEncoding` automatically discovers all applicable $\mathbb{Z}_2$ symmetry generators, with no manual input. We can inspect the labelled list:

In [3]:
for label in encoding.symmetry_generator_labels:
    print(' ', label)

  P↑
  P↓
  T_(a0)/2
  T_(a1)/2
  T_(a2)/2
  σ[+00]
  σ[0+0]
  σ[00+]


The labels group naturally into three families:

- `P↑`, `P↓` — the **spin parities** of the active-space electron count. These are the same as in the molecular case.
- `T_(a0)/2`, `T_(a1)/2`, `T_(a2)/2` — three independent **half-lattice crystal translations**, one along each Cartesian primitive lattice vector. These are the new ingredient compared to molecular SAE.
- `σ[+00]`, `σ[0+0]`, `σ[00+]` — three independent **mirror planes** of the cubic $Pm\overline{3}m$ point group. One of them is shown below as a translucent purple plane cutting obliquely through the supercell:

<p align="center" style="background-color:white; padding:8px;"><img src="figures_periodic/supercell_cscl_mirror_plane.png" alt="mirror plane of the CsCl supercell" width="380"/></p>

Spin parity contributes 2 generators, supercell point-group operations contribute 3, and crystal translations contribute 3, for a total of **8 generators** and 8 qubit removals: $14 \rightarrow 6$. This is the theoretical maximum reduction available on a $(2,2,2)$ supercell — every type of $\mathbb{Z}_2$ symmetry that can survive a periodic Hamiltonian (spin, point group, half translation) is present, which is what makes CsCl a useful headline demonstration.

<a name="hamiltonian"></a>
## The reduced Hamiltonian and its spectrum

The qubit-reduced Hamiltonian is accessible through the same `.hamiltonian` property as in the molecular case:

In [4]:
from openfermion.linalg import qubit_operator_sparse

H = encoding.hamiltonian
n_terms = len(list(H.terms))
print(f'Reduced Hamiltonian: {n_terms} Pauli terms on '
      f'{encoding.nspinorbital - len(encoding.target_qubits)} qubits')

H_dense = qubit_operator_sparse(H).toarray()
eigs = np.sort(np.linalg.eigvalsh(H_dense))
print(f'Ground state energy : {eigs[0]:.10f} Ha')
print(f'First excited state : {eigs[1]:.10f} Ha')

Reduced Hamiltonian: 71 Pauli terms on 6 qubits
Ground state energy : -279.0523770636 Ha
First excited state : -278.4338246985 Ha


To see what the active space looks like in real space, here is an isosurface of MO 62 — the highest-energy occupied orbital inside the active window, transforming as the $x$-component of the $\Gamma_4^-$ HOMO triplet. Red and blue lobes correspond to opposite signs of the orbital amplitude:

<p align="center" style="background-color:white; padding:8px;"><img src="figures_periodic/orbital_cscl_mo62.png" alt="HOMO of CsCl supercell — MO 62" width="380"/></p>

The orbital is centred on the Cl sublattice and changes sign along the $x$ direction — a pattern that maps onto itself, up to a global sign, under each half-lattice translation $\mathbf{T}$ shown above. That is exactly what makes $\mathbf{T}$ act as a Pauli $Z$-like operator on the active-space qubits, and hence a Boolean symmetry of the encoded Hamiltonian.

The reduced Hamiltonian for CsCl CAS(6,7) in the $(2,2,2)$ supercell has 71 Pauli terms on 6 qubits — small enough to fit comfortably on near-term hardware, yet capturing the full CAS(6,7) correlation energy of the periodic system to the accuracy of the active space.

The PeriodicEncoding object exposes the same interface as the molecular `Encoding` — `apply` to map fermionic operators, `qiskit_mapper` to convert excitation operators in a UCC ansatz, and so on — so the VQE workflow from tutorial [05_VQE_circuits](05_VQE_circuits.ipynb) carries over without modification to the crystalline setting.

<p style="text-align: left"> <a href="07_bravyi_kitaev.ipynb" />< Previous: The Bravyi-Kitaev mapping</a> </p>